# Perivascular and subarachnoid space → a 3D layer (MICrONS minnie65)

Søren's tracing `008 - All perivascular spaces.xyz` (nm; line segments in z-sections): the **glia limitans and the arachnoid mater** -- long thin stretches along the pia -- and the **perivascular spaces around the penetrating vessels**, smaller loops in the same sections.

Per section the notebook:
1. joins the segments into curves;
2. fills **between the two long pial curves** (arachnoid above, glia limitans below) -- the subarachnoid / pial perivascular space;
3. fills the **closed loops** around penetrating vessels (open ones are closed);
4. **subtracts the Wan et al. (TriSAM) vessel lumen**, cut at the same section, so the vessels inside are not counted as perivascular space.

Then it stacks the sections into a 1 µm grid (closing gaps of up to 3 µm between traced sections), makes a surface, and measures volumes and each nucleus's distance to the nearest perivascular space.

Outputs in `MyDrive/microns_pvs/`: `pvs_mesh.bin` (same format as the vessel mesh: uint32 n, float32 xyz nm, uint32 triangles), `pvs_summary.json`, `nuc_pvs_dist.csv`, `pvs_qc.png`. Put them in the repo folder `data/pvs/v1/`.


In [ ]:
!pip -q install trimesh


In [ ]:
# ── settings ─────────────────────────────────────────────────────────────
OUT_DIR = "/content/drive/MyDrive/microns_pvs"
USE_DRIVE = True
XYZ_NAME = "008 - All perivascular spaces.xyz"
TRISAM_URL = "https://bossdb-open-data.s3.amazonaws.com/wei2024/minnie/bv/mesh/1%3A0%3A276-1562_240-1133_247-871"
RES = 1.0            # µm per voxel (x, y and z)
LONG_UM = 80.0       # an open curve wider than this is a pial curve (glia limitans / arachnoid)
MAX_PIAL_UM = 40.0   # never fill more than this between two pial curves; beyond it each curve gets PIAL_MIN_UM
PIAL_MIN_UM = 1.5    # thickness given to a pial curve with no partner
CLOSE_GAP_UM = 15.0  # an open small curve whose ends are closer than this is closed
Z_GAP_UM = 3         # gaps between traced sections up to this are closed in z
MC_STEP = 2          # marching-cubes step (1 = full detail, larger = fewer triangles)
LUMEN_PAD_UM = 0.0   # grow the lumen by this before subtracting (e.g. 0.5 for the endothelium)
NUC_CSV = "nucleus_detection_v0.csv"   # for distances; skipped if not found

import os, json, math, time
import numpy as np
if USE_DRIVE:
    try:
        from google.colab import drive; drive.mount("/content/drive")
    except Exception as e:
        print("Drive not mounted (" + str(e) + ")"); OUT_DIR = "/content/microns_pvs"
else:
    OUT_DIR = "/content/microns_pvs"
os.makedirs(OUT_DIR, exist_ok=True)
print("results go to", OUT_DIR)


In [ ]:
# ── the tracing ──────────────────────────────────────────────────────────
path = next((p for p in [os.path.join(OUT_DIR, XYZ_NAME), "/content/" + XYZ_NAME] if os.path.exists(p)), None)
if path is None:
    from google.colab import files
    print("Upload", XYZ_NAME); up = files.upload(); name = next(iter(up))
    path = os.path.join(OUT_DIR, XYZ_NAME); open(path, "wb").write(up[name])
P = np.loadtxt(path) / 1000.0                    # nm -> µm
if len(P) % 2: P = P[:-1]
S = P.reshape(-1, 2, 3)
S = S[np.linalg.norm(S[:, 1] - S[:, 0], axis=1) > 1e-6]
zkey = np.round(S[:, 0, 2], 3)
levels = np.unique(zkey)
lo = S.reshape(-1, 3).min(0) - 10; hi = S.reshape(-1, 3).max(0) + 10
O = np.floor(lo / RES) * RES
NX, NY, NZ = [int(math.ceil((hi[i] - O[i]) / RES)) + 1 for i in range(3)]
print(len(S), "segments in", len(levels), "sections; grid", NX, "x", NY, "x", NZ, "at", RES, "µm =", round(NX * NY * NZ / 1e6), "M voxels")


In [ ]:
# ── the vessels (Wan et al., TriSAM) ─────────────────────────────────────
import urllib.request
vp = os.path.join(OUT_DIR, "trisam_mesh.bin")
if not os.path.exists(vp):
    print("downloading the TriSAM vessel mesh (~50 MB)…"); urllib.request.urlretrieve(TRISAM_URL, vp)
buf = open(vp, "rb").read()
n = np.frombuffer(buf, np.uint32, 1)[0]
V = np.frombuffer(buf, np.float32, n * 3, 4).reshape(-1, 3) / 1000.0     # nm -> µm
F = np.frombuffer(buf, np.uint32, offset=4 + n * 12).reshape(-1, 3)
T = V[F]                                            # (m, 3, 3)
tz0, tz1 = T[:, :, 2].min(1), T[:, :, 2].max(1)
order = np.argsort(tz0); tz0s = tz0[order]
print(len(V), "vertices,", len(F), "triangles; vessel z", round(V[:, 2].min()), "–", round(V[:, 2].max()), "µm")
def lumen_segments(z):
    # the triangles crossing the plane z, and where they cross it: segments (k, 2, 2) in xy
    k = order[: np.searchsorted(tz0s, z, side="right")]
    k = k[tz1[k] >= z]
    if not len(k): return np.zeros((0, 2, 2))
    t = T[k]; d = t[:, :, 2] - z
    pts = np.zeros((len(t), 3, 2)); ok = np.zeros((len(t), 3), bool)
    for e, (a, b) in enumerate(((0, 1), (1, 2), (2, 0))):
        da, db = d[:, a], d[:, b]; m = (da * db) < 0
        f = np.where(m, da / np.where(m, da - db, 1), 0)
        pts[:, e] = t[:, a, :2] + f[:, None] * (t[:, b, :2] - t[:, a, :2]); ok[:, e] = m
    keep = ok.sum(1) == 2
    if not keep.any(): return np.zeros((0, 2, 2))
    o = np.argsort(~ok[keep], axis=1, kind="stable")[:, :2]
    return np.take_along_axis(pts[keep], o[:, :, None], axis=1)


In [ ]:
# ── filling one section ──────────────────────────────────────────────────
def scan_fill(segs, mask):
    # even-odd fill of closed loops given as a soup of segments (k, 2, 2), µm
    if not len(segs): return
    y0 = segs[:, :, 1].min(1); y1 = segs[:, :, 1].max(1)
    jmin = max(0, int((y0.min() - O[1]) / RES)); jmax = min(NY - 1, int((y1.max() - O[1]) / RES) + 1)
    for j in range(jmin, jmax + 1):
        yc = O[1] + (j + 0.5) * RES
        m = (np.minimum(segs[:, 0, 1], segs[:, 1, 1]) <= yc) & (np.maximum(segs[:, 0, 1], segs[:, 1, 1]) > yc)
        if not m.any(): continue
        s = segs[m]; f = (yc - s[:, 0, 1]) / (s[:, 1, 1] - s[:, 0, 1]); x = np.sort(s[:, 0, 0] + f * (s[:, 1, 0] - s[:, 0, 0]))
        for a, b in zip(x[0::2], x[1::2]):
            i0 = max(0, int(math.ceil((a - O[0]) / RES - 0.5))); i1 = min(NX - 1, int(math.floor((b - O[0]) / RES - 0.5)))
            if i1 >= i0: mask[j, i0:i1 + 1] ^= True
def draw_line(segs, mask, half):
    for (x0, y0), (x1, y1) in segs:
        L = max(1e-6, math.hypot(x1 - x0, y1 - y0)); nstep = int(L / (RES * 0.5)) + 1
        for t in np.linspace(0, 1, nstep + 1):
            x, y = x0 + t * (x1 - x0), y0 + t * (y1 - y0)
            i0, i1 = int((x - half - O[0]) / RES), int((x + half - O[0]) / RES); j0, j1 = int((y - half - O[1]) / RES), int((y + half - O[1]) / RES)
            mask[max(0, j0):min(NY, j1 + 1), max(0, i0):min(NX, i1 + 1)] = True
def components(segs):
    # join segments sharing end points (within 0.05 µm) into curves
    key = lambda p: (round(p[0] * 20), round(p[1] * 20))
    parent = list(range(len(segs)))
    def find(a):
        while parent[a] != a: parent[a] = parent[parent[a]]; a = parent[a]
        return a
    at = {}
    for i, (a, b) in enumerate(segs):
        for p in (a, b):
            k = key(p)
            if k in at: parent[find(i)] = find(at[k])
            else: at[k] = i
    groups = {}
    for i in range(len(segs)): groups.setdefault(find(i), []).append(i)
    out = []
    for g in groups.values():
        s = segs[g]; deg = {}
        for a, b in s:
            for p in (a, b): deg[key(p)] = deg.get(key(p), 0) + 1
        ends = [k for k, v in deg.items() if v == 1]
        out.append({"segs": s, "closed": len(ends) == 0, "ends": ends, "w": np.ptp(s[:, :, 0]), "h": np.ptp(s[:, :, 1])})
    return out
def section_mask(segs2, z, stats):
    mask = np.zeros((NY, NX), bool)
    comps = components(segs2)
    pial = [c for c in comps if not c["closed"] and c["w"] > LONG_UM]
    rest = [c for c in comps if c["closed"] or c["w"] <= LONG_UM]
    # the pial space: between the curves, column by column
    if pial:
        # every (column, y, curve) where a pial curve crosses a pixel column, all at once
        cols, ys, cs = [], [], []
        for k, c in enumerate(pial):
            for (x0, y0), (x1, y1) in c["segs"]:
                if x1 < x0: x0, y0, x1, y1 = x1, y1, x0, y0
                i0 = int(math.ceil((x0 - O[0]) / RES - 0.5)); i1 = int(math.ceil((x1 - O[0]) / RES - 0.5)) - 1
                if i1 < i0: continue
                ii = np.arange(i0, i1 + 1); xc = O[0] + (ii + 0.5) * RES
                cols.append(ii); ys.append(y0 + (xc - x0) / max(x1 - x0, 1e-9) * (y1 - y0)); cs.append(np.full(len(ii), k))
        if cols:
            cols = np.concatenate(cols); ys = np.concatenate(ys); cs = np.concatenate(cs)
            o = np.argsort(cols, kind="stable"); cols, ys, cs = cols[o], ys[o], cs[o]
            starts = np.r_[0, np.nonzero(np.diff(cols))[0] + 1, len(cols)]
            for a, b in zip(starts[:-1], starts[1:]):
                i = cols[a]
                if i < 0 or i >= NX: continue
                yy = ys[a:b]
                if len(set(cs[a:b].tolist())) >= 2 and yy.max() - yy.min() <= MAX_PIAL_UM: lohi = [(yy.min(), yy.max())]; stats["pial_cols"] += 1
                else: lohi = [(y - PIAL_MIN_UM / 2, y + PIAL_MIN_UM / 2) for y in yy]
                for y0_, y1_ in lohi:
                    j0 = max(0, int((y0_ - O[1]) / RES)); j1 = min(NY - 1, int((y1_ - O[1]) / RES))
                    if j1 >= j0: mask[j0:j1 + 1, i] = True
    # the loops around penetrating vessels
    loops = []
    for c in rest:
        s = c["segs"]
        if not c["closed"]:
            if len(c["ends"]) == 2:
                a = np.array(c["ends"][0]) / 20.0; b = np.array(c["ends"][1]) / 20.0
                if np.hypot(*(a - b)) <= CLOSE_GAP_UM: s = np.concatenate([s, [[a, b]]]); stats["closed_by_us"] += 1
                else: draw_line(s, mask, PIAL_MIN_UM / 2); stats["open_drawn"] += 1; continue
            else: draw_line(s, mask, PIAL_MIN_UM / 2); stats["open_drawn"] += 1; continue
        loops.append(s)
    if loops:
        lm = np.zeros_like(mask); scan_fill(np.concatenate(loops), lm); mask |= lm; stats["loops"] += len(loops)
    # minus the vessel lumen
    ls = lumen_segments(z)
    if len(ls):
        lum = np.zeros_like(mask); scan_fill(ls, lum)
        if LUMEN_PAD_UM > 0:
            from scipy.ndimage import binary_dilation
            lum = binary_dilation(lum, iterations=int(round(LUMEN_PAD_UM / RES)))
        stats["lumen_removed"] += int((mask & lum).sum()); mask &= ~lum
    return mask


In [ ]:
# ── all sections → a 1 µm volume ─────────────────────────────────────────
vol = np.zeros((NZ, NY, NX), bool)
stats = {"pial_cols": 0, "closed_by_us": 0, "open_drawn": 0, "loops": 0, "lumen_removed": 0}
t0 = time.time(); qc = []
for n_, z in enumerate(levels):
    segs2 = S[zkey == z][:, :, :2]
    m = section_mask(segs2, z, stats)
    k = int(round((z - O[2]) / RES)); vol[min(NZ - 1, max(0, k))] |= m
    if n_ in (len(levels) // 5, len(levels) // 2, 4 * len(levels) // 5): qc.append((z, segs2, m))
    if n_ % 100 == 0: print(f"section {n_ + 1}/{len(levels)}  ({time.time() - t0:.0f} s)")
from scipy.ndimage import binary_closing
before = int(vol.sum())
if Z_GAP_UM > 0:
    vol = binary_closing(vol, structure=np.ones((Z_GAP_UM * 2 + 1, 1, 1), bool))
print("filled voxels:", before, "->", int(vol.sum()), "after closing z-gaps;", stats)


In [ ]:
# ── surface ──────────────────────────────────────────────────────────────
from skimage.measure import marching_cubes
verts, faces, nv = [], [], 0
CH = 64
for z0 in range(0, NZ, CH):
    sub = vol[max(0, z0 - 1): min(NZ, z0 + CH + 1)].astype(np.uint8)
    if sub.sum() == 0 or sub.shape[0] < 2: continue
    sub = np.pad(sub, 1)
    try: v, f, _, _ = marching_cubes(sub, 0.5, step_size=MC_STEP, allow_degenerate=False)
    except Exception as e: print("chunk", z0, e); continue
    v = v - 1; v[:, 0] += max(0, z0 - 1)
    # keep each chunk's own z range so the overlaps are not doubled
    keep = (v[f][:, :, 0].mean(1) >= z0 - 0.5) & (v[f][:, :, 0].mean(1) < z0 + CH - 0.5)
    f = f[keep]
    xyz = np.stack([O[0] + v[:, 2] * RES, O[1] + v[:, 1] * RES, O[2] + v[:, 0] * RES], 1) * 1000.0     # µm -> nm
    verts.append(xyz.astype(np.float32)); faces.append((f + nv).astype(np.uint32)); nv += len(xyz)
Vm = np.concatenate(verts); Fm = np.concatenate(faces)
with open(os.path.join(OUT_DIR, "pvs_mesh.bin"), "wb") as fh:
    fh.write(np.uint32(len(Vm)).tobytes()); fh.write(Vm.tobytes()); fh.write(Fm.tobytes())
print(len(Vm), "vertices,", len(Fm), "triangles,", round(os.path.getsize(os.path.join(OUT_DIR, "pvs_mesh.bin")) / 1e6, 1), "MB")


In [ ]:
# ── volumes, distances, QC ───────────────────────────────────────────────
import pandas as pd
summary = {"voxel_um": RES, "volume_um3": float(vol.sum() * RES ** 3), "sections": int(len(levels)), "origin_um": O.tolist(),
           "shape_zyx": [NZ, NY, NX], **{k: int(v) for k, v in stats.items()}}
# distance from every nucleus to the nearest perivascular space (2 µm grid)
cand = [os.path.join(OUT_DIR, NUC_CSV), "/content/" + NUC_CSV, "/content/drive/MyDrive/microns_m1300_roots/" + NUC_CSV]
np_ = next((p for p in cand if os.path.exists(p)), None)
if np_:
    from scipy.ndimage import distance_transform_edt
    z2, y2, x2 = NZ // 2, NY // 2, NX // 2
    small = vol[: z2 * 2, : y2 * 2, : x2 * 2].reshape(z2, 2, y2, 2, x2, 2).any(axis=(1, 3, 5))
    D = distance_transform_edt(~small) * 2 * RES
    nuc = pd.read_csv(np_, dtype=str, engine="python", on_bad_lines="skip")
    nuc = nuc[nuc["id"].str.fullmatch(r"\d+", na=False)]
    xyz = nuc[["pt_position_x", "pt_position_y", "pt_position_z"]].astype(float).values / 1000.0
    g = np.round((xyz - O) / (2 * RES)).astype(int)[:, ::-1]                     # z, y, x
    ok = np.all((g >= 0) & (g < np.array(D.shape)), 1)
    d = np.full(len(xyz), np.nan); d[ok] = D[g[ok, 0], g[ok, 1], g[ok, 2]]
    pd.DataFrame({"nucleus_id": nuc["id"].values, "pvs_dist_um": np.round(d, 1)}).to_csv(os.path.join(OUT_DIR, "nuc_pvs_dist.csv"), index=False)
    summary["nuclei_with_distance"] = int(ok.sum()); print("distances for", int(ok.sum()), "nuclei")
else:
    print("no nucleus table found - distances skipped (put", NUC_CSV, "in", OUT_DIR, ")")
json.dump(summary, open(os.path.join(OUT_DIR, "pvs_summary.json"), "w"), indent=1)
import matplotlib; import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
fig, ax = plt.subplots(1, len(qc), figsize=(7 * len(qc), 6))
for A, (z, segs2, m) in zip(np.atleast_1d(ax), qc):
    A.imshow(m, origin="upper", extent=[O[0], O[0] + NX * RES, O[1] + NY * RES, O[1]], cmap="Blues")
    A.add_collection(LineCollection(segs2, lw=0.6, colors="k")); A.set_title(f"z = {z:.1f} µm"); A.set_aspect("equal")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "pvs_qc.png"), dpi=90)
print(json.dumps(summary, indent=1))


In [ ]:
# ── if Drive was not mounted, hand the files to the browser ──────────────
# (2026-10-21: on the first real run the mount failed and the results stayed in the Colab session.)
if not OUT_DIR.startswith("/content/drive"):
    from google.colab import files
    for f in ["pvs_mesh.bin", "pvs_summary.json", "nuc_pvs_dist.csv"]:
        p = os.path.join(OUT_DIR, f)
        if os.path.exists(p): files.download(p)
    print("downloaded -- put them in microns-neuroglancer-tool/data/pvs/v1/")
